In [1]:
%cd ..

/home/ricka/Git/GitHub/RickArko/kaggle/playground/kcom-store-sales


# Linear Regression Baseline — Store Sales

Illustrative walkthrough of the **Ridge** baseline inspired by Kaggle's
[Linear Regression with Time Series](https://www.kaggle.com/code/ryanholbrook/linear-regression-with-time-series) course.

This notebook trains a global Ridge model on a **5-store smoke subsample** so it
runs in seconds. For the full dataset with log-transform, use:

```bash
make train-linear CONFIG=config/linear-log.yaml RUN_NAME=my-log-ridge
```

In [2]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge

from store_sales.data import load_config, load_data, merge_tables, timeseries_split
from store_sales.features import TimeSeriesFeatureEngineer
from store_sales.metrics import rmsle
from store_sales.models import TimeSeriesModel

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

CONFIG_PATH = "config/linear-smoke.yaml"
cfg = load_config(CONFIG_PATH)
print(f"Config: {CONFIG_PATH}  (run_scope={cfg.get('run_scope', '?')})")

Config: config/linear-smoke.yaml  (run_scope=smoke)


## 1. Load & merge data

We join store metadata, oil prices, holidays, and transactions onto the train/test
frames — the same pipeline used by `scripts/train_linear.py`.

In [3]:
tables = load_data(cfg["paths"]["data"])
train, test = merge_tables(tables)

# Smoke subsample: 5 stores
max_stores = cfg.get("subsample", {}).get("max_stores", 5)
keep_stores = sorted(train["store_nbr"].unique())[:max_stores]
train = train[train["store_nbr"].isin(keep_stores)].copy().reset_index(drop=True)
test = test[test["store_nbr"].isin(keep_stores)].copy().reset_index(drop=True)

target_col = cfg["competition"]["target"]
y_full = train[target_col].copy()
print(f"Train: {train.shape}  Test: {test.shape}")
print(f"Stores: {sorted(train['store_nbr'].unique())}")
print(f"Date range: {train['date'].min()} → {train['date'].max()}")

Train: (277860, 13)  Test: (2640, 12)
Stores: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Date range: 2013-01-01 00:00:00 → 2017-08-15 00:00:00


## 2. Feature engineering

**Critical rule:** lag and rolling features must be computed on the full sorted
store-family history *before* the train/validation split.

Features used:
- **Date**: year, month, dayofweek, time_step (trend)
- **Lags**: sales from 1, 7, 14, 28 days ago
- **Rolling**: mean/std/min/max over 7/14/28/56-day windows
- **External**: onpromotion, oil price, holidays, transactions

In [4]:
feat_cfg = cfg["features"]
engineer = TimeSeriesFeatureEngineer(
    date_col=feat_cfg.get("date_col", "date"),
    store_col=feat_cfg.get("store_col", "store_nbr"),
    family_col=feat_cfg.get("family_col", "family"),
    onpromotion_col=feat_cfg.get("onpromotion_col", "onpromotion"),
    date_features=feat_cfg.get("date_features", []),
    drop_cols=feat_cfg.get("drop_cols", []),
    lag_config=feat_cfg.get("lag_features", []),
    rolling_config=feat_cfg.get("rolling_features", []),
    ref_date=train["date"].min(),
)

X_lag, X_test_feat = engineer.create_lag_features(train, test, target_col)
engineer.fit(X_lag)
print(f"Feature matrix (pre-split): {X_lag.shape}")
print(f"Sample columns: {list(X_lag.columns[:8])} ...")

Feature matrix (pre-split): (277695, 33)
Sample columns: ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state'] ...


## 3. Train / validation split

Hold out the last 16 days as validation — mimics the competition's 16-day test horizon.

In [5]:
val_days = cfg.get("timeseries", {}).get("test_period_days", 16)
X_train_raw, X_val_raw = timeseries_split(X_lag, val_days)
y_train = y_full.loc[X_train_raw.index]
y_val = y_full.loc[X_val_raw.index]

X_train = engineer.transform(X_train_raw)
X_val = engineer.transform(X_val_raw)
X_test = engineer.transform(X_test_feat)

# One-hot encode store & family for linear regression
cat_cols = [c for c in X_train.columns if X_train[c].dtype.name == "category"]
known_cats = {c: sorted(X_train[c].cat.categories.tolist()) for c in cat_cols}
for df in (X_train, X_val, X_test):
    for c in cat_cols:
        df[c] = pd.Categorical(df[c], categories=known_cats[c])
    pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int, inplace=True)

train_cols = list(X_train.columns)
for df in (X_val, X_test):
    for c in set(train_cols) - set(df.columns):
        df[c] = 0
    df.drop(columns=[c for c in df.columns if c not in train_cols], inplace=True)
X_val = X_val[train_cols]
X_test = X_test[train_cols]
X_train.fillna(0, inplace=True)
X_val.fillna(0, inplace=True)
X_test.fillna(0, inplace=True)

val_split_date = sorted(train["date"].unique())[-val_days]
print(f"Train rows: {len(X_train)}  Val rows: {len(X_val)}  Features: {X_train.shape[1]}")
print(f"Validation starts: {val_split_date}")

KeyError: '[165, 330, 495, 660, 825, 990, 1155, 1320, 1485, 1650, 1815, 1980, 2310, 2475, 2640, 2805, 2970, 3135, 3300, 3465, 3630, 3795, 3960, 4125, 4455, 4620, 4785, 4950, 5115, 5280, 5445, 5610, 5775, 5940, 6105, 6270, 6600, 6765, 6930, 7095, 7260, 7425, 7590, 7755, 7920, 8085, 8250, 8415, 8745, 9075, 9240, 9405, 9570, 9735, 9900, 10065, 10230, 10395, 10560, 10725, 10890, 11220, 11385, 11550, 11715, 11880, 12045, 12210, 12375, 12540, 12705, 12870, 13035, 13365, 13530, 13695, 13860, 14025, 14190, 14355, 14520, 14685, 14850, 15015, 15180, 15510, 15675, 15840, 16005, 16170, 16335, 16500, 16665, 16830, 16995, 17160, 17325, 17655, 17985, 18150, 18315, 18480, 18645, 18810, 18975, 19140, 19305, 19470, 19635, 19800, 20130, 20295, 20460, 20625, 20790, 20955, 21120, 21285, 21450, 21615, 21780, 21945, 22275, 22440, 22605, 22770, 22935, 23100, 23265, 23430, 23595, 23760, 23925, 24090, 24420, 24585, 24750, 24915, 25080, 25245, 25410, 25575, 25740, 25905, 26070, 26235, 26565, 26895, 27060, 27225,

## 4. Fit Ridge regression

A single global model across all store-family series. Ridge (L2) regularisation
prevents overfitting with 100+ one-hot + lag features.

In [ ]:
alpha = cfg["model"].get("alpha", 0.5)
ridge = Ridge(alpha=alpha, fit_intercept=True, random_state=42)
ts_model = TimeSeriesModel(ridge)
ts_model.fit(X_train, y_train)

val_preds = np.maximum(ts_model.predict(X_val), 0)
val_score = rmsle(y_val.values, val_preds)
print(f"Validation RMSLE: {val_score:.4f}")
print(f"(Full-dataset log-Ridge typically scores ~0.42 — see make compare)")

## 5. Visualise one series

Plot actual vs predicted for the highest-volume store-family combination.

In [ ]:
# Pick hero series (highest total sales)
totals = train.groupby(["store_nbr", "family"])["sales"].sum().sort_values(ascending=False)
hero_store, hero_family = totals.index[0]

meta = X_lag[["date", "store_nbr", "family"]].copy()
meta["actual"] = y_full.loc[X_lag.index].values
all_preds = np.maximum(ts_model.predict(engineer.transform(X_lag)), 0)
meta["predicted"] = all_preds

hero = meta[(meta["store_nbr"] == hero_store) & (meta["family"] == hero_family)].sort_values("date")
val_start = pd.Timestamp(val_split_date)
hero = hero[hero["date"] >= val_start - pd.Timedelta(days=90)]

fig, ax = plt.subplots(figsize=(14, 4.5))
pre_val = hero[hero["date"] < val_start]
post_val = hero[hero["date"] >= val_start]

ax.plot(pre_val["date"], pre_val["actual"], color="#aaaaaa", linewidth=0.6, alpha=0.7, label="Train actual")
ax.plot(post_val["date"], post_val["actual"], color="#1f77b4", linewidth=1.2, label="Val actual")
ax.plot(post_val["date"], post_val["predicted"], color="#ff7f0e", linewidth=1.2, linestyle="--", label="Val predicted")
ax.axvline(val_start, color="black", linewidth=0.8, linestyle=":", alpha=0.5)
ax.set_title(f"Store {hero_store} — {hero_family}  (val RMSLE={val_score:.4f})", fontweight="bold")
ax.set_ylabel("Sales")
ax.legend(fontsize=8)
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

## 6. Daily aggregate view

Sum predictions across all stores to see the big-picture fit during validation.

In [ ]:
daily = meta.groupby("date").agg(actual=("actual", "sum"), predicted=("predicted", "sum")).reset_index()
daily = daily[daily["date"] >= val_start - pd.Timedelta(days=60)]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily["date"], daily["actual"], color="black", linewidth=1.0, label="Actual")
ax.plot(daily["date"], daily["predicted"], color="#ff7f0e", linewidth=0.9, linestyle="--", label="Ridge predicted")
ax.axvline(val_start, color="gray", linestyle=":", alpha=0.6)
ax.set_title("Daily aggregate sales — smoke subsample", fontweight="bold")
ax.set_ylabel("Total sales")
ax.legend()
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

## Next steps

| Command | What it does |
|---|---|
| `make train-linear CONFIG=config/linear-log.yaml RUN_NAME=demo` | Full-dataset log-Ridge (~30s, RMSLE ~0.42) |
| `make benchmark-linear` | Compare all linear variants side-by-side |
| `make viz-gif` | Render animated pipeline GIF for README |
| `notebooks/visualize_predictions.ipynb` | Multi-model comparison plots |